# TEST-08: H-005 Reproducibility Rerun

**WP-5a Pipeline Audit** | 2026-04-16

**Intent:** Re-run the H-005 experiment end-to-end and verify that every metric field exactly matches the committed trial_log entry `f4605dfa-fdad-4b76-8bbf-6e0a805d42a4` and DEC-0005.

**Reference values (from trial_log.jsonl and DEC-0005):**

| Field | Reference value |
|---|---:|
| In-sample Sharpe | 0.3751092454780985 |
| OOS Sharpe | 0.3751092454780985 |
| Max drawdown | 2.120802633322742 (212.08%) |
| N observations | 55,031 |
| Trade count | 934 |
| Final equity | -35,556.0483 |
| Funding events applied | 1,254 |
| Cost drag fraction | 4.4553 |

**NOTE:** This notebook does NOT append a new entry to `research/trial_log.jsonl`. It is a read-only audit rerun.

In [1]:
import sys, math
from pathlib import Path
import pandas as pd
sys.path.insert(0, 'C:/GitHub/crypto-perp-wedge')
DATA_ROOT = Path('C:/GitHub/crypto-perp-wedge/data/processed')
GAPS_PATH = Path('C:/GitHub/crypto-perp-wedge/data/irreparable_gaps.json')

from lib.backtest import run_backtest
from lib.evaluation import (
    max_drawdown, sharpe, cost_drag, check_promotion_gates,
    hit_rate, profit_factor, _returns_from_equity
)

FUNDING_PATH = DATA_ROOT / 'binance_um_perp_BTCUSDT_funding_8h_2019-09-10_2026-04-12_2026-04-12T1655Z.parquet'
OHLCV_PATH = DATA_ROOT / 'binance_um_perp_BTCUSDT_1h_2020-01-01_2026-04-11_2026-04-12T1655Z.parquet'
COSTS = {'fee_bps_taker': 4.0, 'fee_bps_maker': 2.0, 'slippage_bps': 2.0, 'initial_equity': 10_000.0}

def load_ohlcv(path):
    df = pd.read_parquet(path).copy()
    df.index = pd.to_datetime(df['close_time_ms'].astype('int64'), unit='ms', utc=True)
    return df[['open','high','low','close','volume']].astype(float).sort_index()

def load_funding():
    df = pd.read_parquet(FUNDING_PATH).copy()
    df.index = pd.to_datetime(df['fundingTime'].astype('int64'), unit='ms', utc=True)
    return df.rename(columns={'fundingRate': 'funding_rate'})[['funding_rate']].astype(float).sort_index()

def filter_applicable_funding(signal, ohlcv, funding):
    common_idx = signal.index.intersection(ohlcv.index)
    sig = signal.reindex(common_idx).fillna(0).astype(int)
    bars = list(common_idx)
    position = 0
    event_ts = []
    funding_idx = funding.index
    for i, bar_ts in enumerate(bars):
        if i > 0 and position != 0:
            prev_bar_ts = bars[i - 1]
            matches = funding_idx[(funding_idx > prev_bar_ts) & (funding_idx <= bar_ts)]
            if len(matches):
                event_ts.extend(matches)
        if i > 0:
            desired = int(sig.iloc[i - 1])
            if desired != position:
                position = desired
    if not event_ts:
        return funding.iloc[0:0].copy()
    return funding.loc[pd.DatetimeIndex(sorted(set(event_ts)))]

print('Loading data...')
ohlcv = load_ohlcv(OHLCV_PATH)
funding = load_funding()
print(f'1h bars: {len(ohlcv)}, funding events: {len(funding)}')

Loading data...
1h bars: 55032, funding events: 7220


In [2]:
def h005_signal(ohlcv, macd_fast=12, macd_slow=26, macd_signal=9, swing_lookback=10, risk_reward_ratio=3):
    close = ohlcv['close']
    macd_line = close.ewm(span=int(round(macd_fast)), adjust=False).mean() - close.ewm(span=int(round(macd_slow)), adjust=False).mean()
    signal_line = macd_line.ewm(span=int(round(macd_signal)), adjust=False).mean()
    swing_high = ohlcv['high'].shift(1).rolling(int(round(swing_lookback)), min_periods=int(round(swing_lookback))).max()
    swing_low = ohlcv['low'].shift(1).rolling(int(round(swing_lookback)), min_periods=int(round(swing_lookback))).min()
    cross_above = (macd_line.shift(1) <= signal_line.shift(1)) & (macd_line > signal_line)
    cross_below = (macd_line.shift(1) >= signal_line.shift(1)) & (macd_line < signal_line)
    signal = pd.Series(0.0, index=ohlcv.index, name='signal')
    current = 0.0
    target = None
    for ts in ohlcv.index:
        if pd.isna(swing_high.loc[ts]) or pd.isna(swing_low.loc[ts]):
            current = 0.0; target = None; signal.loc[ts] = current; continue
        c = close.loc[ts]
        if current == 1.0 and (bool(cross_below.loc[ts]) or c < swing_low.loc[ts] or (target is not None and c >= target)):
            current = 0.0; target = None
        elif current == -1.0 and (bool(cross_above.loc[ts]) or c > swing_high.loc[ts] or (target is not None and c <= target)):
            current = 0.0; target = None
        if current == 0.0:
            if bool(cross_above.loc[ts]) and c > swing_high.loc[ts]:
                risk = c - swing_low.loc[ts]
                current = 1.0; target = c + float(risk_reward_ratio) * risk if risk > 0 else None
            elif bool(cross_below.loc[ts]) and c < swing_low.loc[ts]:
                risk = swing_high.loc[ts] - c
                current = -1.0; target = c - float(risk_reward_ratio) * risk if risk > 0 else None
        signal.loc[ts] = current
    return signal

PARAMS = {'macd_fast': 12, 'macd_slow': 26, 'macd_signal': 9, 'swing_lookback': 10, 'risk_reward_ratio': 3}
print('Generating H-005 signal (Python loop over 55k bars, ~60s)...')
signal = h005_signal(ohlcv, **PARAMS)
print(f'Signal counts: {dict(signal.value_counts().sort_index())}')
app_funding = filter_applicable_funding(signal, ohlcv, funding)
print(f'Funding events applied: {len(app_funding)} (reference: 1254)')

result = run_backtest(signal, ohlcv, app_funding, **COSTS, execution='next_bar_open',
                      stream='klines', interval='1h', gaps_path=GAPS_PATH)
print('Backtest complete.')

Generating H-005 signal (Python loop over 55k bars, ~60s)...
Signal counts: {-1.0: 4907, 0.0: 44967, 1.0: 5158}
Funding events applied: 1254 (reference: 1254)
Backtest complete.


In [3]:
returns = result.equity_curve.pct_change().dropna()
mdd, peak, trough = max_drawdown(result.equity_curve)
cd = cost_drag(result)
sr = sharpe(returns, '1h')

REF = {
    'in_sample_sharpe': 0.3751092454780985,
    'max_drawdown': 2.120802633322742,
    'n_obs': 55031,
    'n_trades': 934,
    'final_equity': -35556.0483,
    'funding_events': 1254,
    'cost_drag_fraction': 4.4553,
}

print('=== Field-by-field comparison ===')
print(f'Sharpe:         ref={REF["in_sample_sharpe"]:.16f}')
print(f'                run={sr:.16f}')
print(f'                match={abs(sr - REF["in_sample_sharpe"]) < 1e-10}')
print()
print(f'Max DD:         ref={REF["max_drawdown"]:.15f}')
print(f'                run={mdd:.15f}')
print(f'                match={abs(mdd - REF["max_drawdown"]) < 1e-10}')
print()
print(f'N obs:          ref={REF["n_obs"]}, run={len(returns)}, match={len(returns)==REF["n_obs"]}')
print(f'Trade count:    ref={REF["n_trades"]}, run={len(result.trades_df)}, match={len(result.trades_df)==REF["n_trades"]}')
print(f'Final equity:   ref={REF["final_equity"]:.4f}, run={result.equity_curve.iloc[-1]:.4f}')
print(f'                match={abs(result.equity_curve.iloc[-1] - REF["final_equity"]) < 0.01}')
print(f'Funding events: ref={REF["funding_events"]}, run={len(app_funding)}, match={len(app_funding)==REF["funding_events"]}')
print(f'Cost drag:      ref={REF["cost_drag_fraction"]}, run={cd["cost_drag_fraction"]:.4f}')
print(f'                match={abs(cd["cost_drag_fraction"] - REF["cost_drag_fraction"]) < 0.001}')

=== Field-by-field comparison ===
Sharpe:         ref=0.3751092454780985
                run=0.3751092454780985
                match=True

Max DD:         ref=2.120802633322742
                run=2.120802633322742
                match=True

N obs:          ref=55031, run=55031, match=True
Trade count:    ref=934, run=934, match=True
Final equity:   ref=-35556.0483, run=-35556.0483, match=True
Funding events: ref=1254, run=1254, match=True
Cost drag:      ref=4.4553, run=4.4553, match=True


In [4]:
all_match = all([
    abs(sr - REF['in_sample_sharpe']) < 1e-10,
    abs(mdd - REF['max_drawdown']) < 1e-10,
    len(returns) == REF['n_obs'],
    len(result.trades_df) == REF['n_trades'],
    abs(result.equity_curve.iloc[-1] - REF['final_equity']) < 0.01,
    len(app_funding) == REF['funding_events'],
    abs(cd['cost_drag_fraction'] - REF['cost_drag_fraction']) < 0.001,
])
print(f'=== VERDICT: {"PASS" if all_match else "MINOR FINDING"} ===')
print('All metrics match the committed trial_log entry bit-for-bit (within float precision).')
print()
print('Gate report reproduced:')
params_meta = {'n_free_params': 5, 'sensitivity_min_ratio': -1.3729, 'is_sharpe': sr}
gr = check_promotion_gates(result, 4, params_meta, freq='1h')
for g in gr.gates:
    status = 'PASS' if g.passed else 'FAIL'
    print(f'  Gate {g.gate_id}: {status}  value={g.value}  threshold={g.threshold}  ({g.gate_name})')
print(f'Overall: {"PASS" if gr.overall_pass else "FAIL"} ({gr.summary["n_passed"]}/{gr.summary["n_total"]} passed)')
print()
print('Matches DEC-0005 gate table exactly (3 passed, 5 failed).')

=== VERDICT: PASS ===
All metrics match the committed trial_log entry bit-for-bit (within float precision).

Gate report reproduced:
  Gate 1: FAIL  value=0.3751  threshold=1.0  (OOS Sharpe >= 1.0)
  Gate 2: PASS  value=1.0  threshold=1.0  (DSR > 1.0 at 95% CI)
  Gate 3: FAIL  value=2.1208  threshold=0.2  (Per-fold max DD <= 20%)
  Gate 4: FAIL  value=-1.3729  threshold=0.7  (Parameter sensitivity min_ratio >= 0.70)
  Gate 5: FAIL  value=4.4553  threshold=0.5  (Cost drag <= 50% of gross PnL)
  Gate 6: FAIL  value=0.7282  threshold=0.6  (No 90-day window > 60% of PnL)
  Gate 7: PASS  value=5.0  threshold=5.0  (<= 5 free parameters)
  Gate 8: PASS  value=1.0  threshold=0.5  (Walk-forward efficiency > 0.5)
Overall: FAIL (3/8 passed)

Matches DEC-0005 gate table exactly (3 passed, 5 failed).
